## PAIR-Scan analysis example notebook:
To calculate PAIR-Scan confidence in TCR-Ag pairs being reactive and pair fold-enrichment, single-well paired TCR- and epitope-barcode sequencing counts generated in [tutorials/pair_scan_counting_tutorial.md](https://github.com/schumacherlab/TCRtoolbox/blob/main/tutorials/pair_scan_counting_tutorial.md) are analyzed in this example notebook. First detected TCR-Ag pairs per well are called from the UMI count table in three steps. TCR and Ag-barcode transcripts are first filtered on a minimum UMI count using `filter_UMI_counts_df_dict` (panel c). If a well contains multiple TCR or multiple Ag-barcode transcripts, non-top transcripts are filtered separately within each transcript type based on their ratio to the well-specific top transcript, using a threshold that is depth-scaled on the top transcript UMI count (panel c). Every remaining pairwise TCR–Ag combination per well is then called as a detected pair, and the number of wells for each pair is counted using `write_tcr_epi_pair_well_count_files` (panel c). Second, expected random TCR–Ag pairing frequencies are estimated from bulk TCR and Ag-barcode library sequencing counts generated in [tutorials/count_bulk_illumina_seq_reads_cli_tutorial.md](https://github.com/schumacherlab/TCRtoolbox/blob/main/tutorials/count_bulk_illumina_seq_reads_cli_tutorial.md). Bulk TCR and Ag read counts are each normalized to sum to 1 and the expected frequency of each TCR–Ag pair due to random pairing is calculated as the product of these values using `calculate_tcr_epi_pair_exp_prob_matrix` (panel e). The observed frequency of each pair, screen fold-enrichment, PAIR-Scan confidence in reactivity, and Benjamini-Hochberg FDR correction of resulting P values are all calculated by `obs_tcr_epi_count_binom_sf_test`, as follows (panel e). The observed frequency of each pair is calculated as its well count divided by the total number of wells with at least one detected TCR–Ag pair. Screen fold-enrichment is calculated as the observed frequency divided by the expected frequency due to random pairing. PAIR-Scan confidence in a pair's reactivity is the probability, under a one-sided binomial test, of observing at least its well count k among the total number of sorted wells with at least a detected TCR–Ag pair n, given its expected pairing frequency p, implemented with Scipy (scipy.stats.binom.sf). Resulting P values are corrected for multiple testing with Benjamini-Hochberg FDR correction, adapted from the statsmodels package. Third, Epi-Epi and TCR-TCR co-occurrence filtering is used to remove TCR–Ag pairs with insufficient independent well support for their TCR, Ag, or both (using `add_tcr_epi_pairs_only_supported_by_co_occurrence_bool_col`, which adds a bool column used for filtering in `write_reactive_pair_results`; panel d). Fourth, only the most significant pair for TCRs that showed high PAIR-Scan confidence with more than one unique Ag is reported, and if a TCR is cross-detected with multiple Ags the remaining less significant pairs are reported as cross-detected pairs for that TCR (`filter_reactive_pairs`, called within `write_reactive_pair_results`).  


![PAIR-Scan analysis overview](pair_scan_analysis_illustration.png)


In [ ]:
import os
from pathlib import Path
import numpy as np
from dotenv import load_dotenv

load_dotenv()
tcr_toolbox_data_path = os.getenv("tcr_toolbox_data_path")

import pandas as pd
import matplotlib as mpl

from tcr_toolbox.sequencing_analysis.plate_tcr_epi_count_analysis import (
    add_tcr_epi_pairs_only_supported_by_co_occurrence_bool_col,
    calculate_tcr_epi_pair_exp_prob_matrix,
    filter_UMI_counts_df_dict,
    init_umi_count_analysis_dir,
    obs_tcr_epi_count_binom_sf_test,
    plot_detected_tcr_epi_plate_map,
    plot_pvalue_corrected_rank_vs_pvalue_corrected,
    plot_tcr_epi_pair_ma_plot,
    read_tcr_epi_pair_counts_csv,
    read_umi_count_tsv,
    write_reactive_pair_results,
    write_tcr_epi_pair_well_count_files,
    add_pair_entropy_over_plate_col
)
from tcr_toolbox.sequencing_analysis.utils import overlap_between_ref_and_count_names, read_gdna_counts_csv
from tcr_toolbox.utils.plot_utils import plot_count_histogram,set_mpl_params
from tcr_toolbox.utils.stat_utils import normalize_counts_df_by_total_counts

mpl, mylines = set_mpl_params(mpl=mpl)

## 1. Sync and initialize analysis directory

### You should be able to sync the following directory structure from a PAIR-Scan counting run:


```text
pair_scan_analysis_example/
├── bulk_seq_data/
│   ├── epi/
│   │   ├── counts/ 
│   │   └── run_logs/
│   └── tcr/
│       ├── counts/
│       └── run_logs/
├── counts/
└── run_logs/
```
#### Further explanation of the files written by the code below is given at the end of the notebook.

In [ ]:
project_dir = Path('/data/pair_scan_analysis_example')

In [ ]:
init_umi_count_analysis_dir(project_dir)

### init_umi_count creates the following directories:


```text
pair_scan_analysis_example/
├── bulk_seq_data/
│   ├── epi/
│   │   ├── counts/
│   │   └── run_logs/
│   └── tcr/
│       ├── counts/
│       └── run_logs/
├── counts/
├── preprocessing/ # created 
├── outs/ # created 
│   └── plate_maps/ # created 
└── run_logs/
```

## 2. Read and filter UMI counts

In [6]:
counts_df_dict = read_umi_count_tsv(project_dir=project_dir, plate_name_split_location=0)

Reading plate: 4
Reading plate: 1
Reading plate: 3
Reading plate: 5
Reading plate: 2


In [ ]:
# Adjust for all your plates: 
epi_umi_threshold_dict = {
  "1":200, 
  "2":150, 
  "3":200,
  "4":150, 
  "5":125, 
}

tcr_umi_threshold_dict = {
      "1":100, 
      "2":100, 
      "3":100, 
      "4":150, 
      "5":100, 
}

min_start_fraction_of_top_umi_count_dict = {
      "1":1.25, 
      "2":1.25, 
      "3":1.25,
      "4":1.25, 
      "5":1.25, 
}

min_end_fraction_of_top_umi_count_dict = {
      "1":0.1, 
      "2":0.1, 
      "3":0.1,
      "4":0.1, 
      "5":0.1, 
}


In [ ]:
# filter_UMI_counts_df_dict writes filter QC plots in preprocessing, use these plots to see whether filter thresholds need to be adjusted. 
# If filter thresholds need to be adjusted, re-run read_umi_count_tsv, adjust filter thresholds above, and re-run this cells. 
counts_df_dict = filter_UMI_counts_df_dict(
    project_dir = project_dir, 
    counts_df_dict = counts_df_dict, 
    epi_umi_threshold=epi_umi_threshold_dict, 
    tcr_umi_threshold=tcr_umi_threshold_dict,
    min_start_fraction_of_top_umi_count=min_start_fraction_of_top_umi_count_dict, 
    min_end_fraction_of_top_umi_count = min_end_fraction_of_top_umi_count_dict, 
    num_bins=50
)

Filtering plate: 4
Number of unique epis: 123
Number of unique tcrs: 101
Filtering plate: 1
Number of unique epis: 120
Number of unique tcrs: 103
Filtering plate: 3
Number of unique epis: 99
Number of unique tcrs: 106
Filtering plate: 5
Number of unique epis: 108
Number of unique tcrs: 95
Filtering plate: 2
Number of unique epis: 115
Number of unique tcrs: 101


In [9]:
plot_detected_tcr_epi_plate_map(project_dir=project_dir, counts_df_dict=counts_df_dict, validating_pair_dict=None, epitope_barcode_length=18, detected_fontsize=3)

Plotting plate: 4
Plotting plate: 1
Plotting plate: 3
Plotting plate: 5
Plotting plate: 2


## 3. Call detected TCR-Epi pairs

In [ ]:
plates_list = ["1", "2", "3", "4", "5"] # A list of plate IDs belonging to one patient is used, as a single pair_scan analysis project directory can contain plates from multiple different patients if those were screened simultaneously.
plates_list_name = "patient-X" # A name is used to label the plates_list, as each pair_scan analysis project dir can hold multiple such lists, one per patient.

write_tcr_epi_pair_well_count_files(
    project_dir=project_dir,
    plates_list=plates_list,
    plates_list_name=plates_list_name,
    counts_df_dict=counts_df_dict.copy(),
    collapse_tcr_technical_duplicates=True, # Set to True as an example here; defaults to False, since all proof-of-concept PAIR-Scan screens in the paper did not use TCR technical duplicates.
    reference_file='/references/150bp_beta_epi_plate.fa', # Set to None if collapse_tcr_technical_duplicates = False
    collapse_epitope_barcode=True,
    epitope_barcode_length=18,
    validating_pair_dict=None,
)

Processing plate: 1
Processing plate: 2
Processing plate: 3
Processing plate: 4
Processing plate: 5
Writing patient-X summary statistics...
Total # of co-culture sort wells counted: 1585


### write_tcr_epi_pair_well_count_files creates a patient-specific directory with detected pair count files: 
```text
pair_scan_analysis_example/
├── bulk_seq_data/
│   ├── epi/
│   │   ├── counts/
│   │   ├── outs/
│   │   └── run_logs/
│   └── tcr/
│       ├── counts/
│       ├── outs/
│       └── run_logs/
├── counts/
├── preprocessing/
├── outs/
│   ├── plate_maps/
│   └── patient-X/ # Created directory in which patient-X detected pair files are written
└── run_logs/
```

## 4. Bulk seq data QC

In [16]:
name_split_location = 0

In [ ]:
bulk_project_dir = project_dir / "bulk_seq_data" / "epi"

In [ ]:
for count_file in (bulk_project_dir / "counts").glob("*.csv"):
    print(count_file.name)

    overlap_between_ref_and_count_names(
        count_file=count_file,
        count_file_ref_name_col="reference_name",
        reference_file='/references/150bp_epi.fa',
        library_ref_id_list=["patient-X"], # only needed if multiple patients in the same reference .fa file (this is useful when screening multiple patient simultaneously)
        print_diff_names=True,
    )

    outs_dir = Path(bulk_project_dir) / "outs"
    outs_dir.mkdir(parents=True, exist_ok=True)

    plot_count_histogram(
        count_file=count_file,
        count_file_ref_name_col="reference_name",
        count_file_count_col="read_count",
        bin_min=0,
        bin_max=7,
        bin_size=0.1,
        log10 = True, 
        xlabel="Number of reads per transcript",
        min_count_for_95th_5th_ratio = 3,
        title=count_file.stem.split("_")[name_split_location],
        save_file_path=outs_dir / f"{count_file.stem.split('_')[name_split_location]}_count_hist.pdf",
        ylim_max=30,
        w=6.25,
        h=6.25,
    )
    print("\n")

bulk_epi_lib_counts.csv
# intersecting: 195
# diff: 0
# union: 195
Sensitivity: 100.00% (195/195 reference names found in counts)
refs not in count names: set()
count names not in ref subset: set()
95th percentile: 372624.25 | 5th percentile: 29047.25 | 95th/5th ratio: 12.83




In [ ]:
bulk_project_dir = project_dir / "bulk_seq_data" / "tcr"

In [ ]:
for count_file in (bulk_project_dir / "counts").glob("*.csv"):
    print(count_file.name)

    overlap_between_ref_and_count_names(
        count_file=count_file,
        count_file_ref_name_col="reference_name",
        reference_file='/references/150bp_beta.fa',
        library_ref_id_list=["patient-X"], # only needed if multiple patients in the same reference .fa file (this is useful when screening multiple patient simultaneously)
        print_diff_names=True,
    )

    outs_dir = Path(bulk_project_dir) / "outs"
    outs_dir.mkdir(parents=True, exist_ok=True)

    plot_count_histogram(
        count_file=count_file,
        count_file_ref_name_col="reference_name",
        count_file_count_col="read_count",
        bin_min=0,
        bin_max=7,
        bin_size=0.1,
        log10 = True, 
        xlabel="Number of reads per transcript",
        min_count_for_95th_5th_ratio = 3,
        title=count_file.stem.split("_")[name_split_location],
        save_file_path=outs_dir / f"{count_file.stem.split('_')[name_split_location]}_count_hist.pdf",
        ylim_max=30,
        w=6.25,
        h=6.25,
    )
    print("\n")

bulk_tcr_lib_counts.csv
# intersecting: 115
# diff: 1
# union: 116
Sensitivity: 99.14% (115/116 reference names found in counts)
refs not in count names: {'1_1_H21_93_patient-X'}
count names not in ref subset: set()
95th percentile: 141875.80 | 5th percentile: 17244.90 | 95th/5th ratio: 8.23




## 5. Calculate expected pair frequency in case of random TCR–neoAg pairing 

In [21]:
plates_list_name = "patient-X"

In [ ]:
count_file = Path(project_dir) / "bulk_seq_data" / "epi" / "counts" / "bulk_epi_lib_counts.csv"
epi_bulk_counts_df = read_gdna_counts_csv(count_file=count_file, count_file_ref_name_col="reference_name", collapse_epitope_barcode=True, epitope_barcode_length=18)
normalized_epi_bulk_counts_df = normalize_counts_df_by_total_counts(count_df=epi_bulk_counts_df, count_col="read_count")

In [ ]:
count_file = Path(project_dir) / "bulk_seq_data" / "tcr" / "counts" / "bulk_tcr_lib_counts.csv"
tcr_bulk_counts_df = read_gdna_counts_csv(
    count_file=count_file,
    count_file_ref_name_col="reference_name",
    collapse_tcr_technical_duplicates=True, # Set to True as an example here; defaults to False, since all proof-of-concept PAIR-Scan screens in the paper did not use TCR technical duplicates.
    reference_file='/references/150bp_beta_epi_plate.fa', # Set to None if collapse_tcr_technical_duplicates = False
)
normalized_tcr_bulk_counts_df = normalize_counts_df_by_total_counts(count_df=tcr_bulk_counts_df, count_col="read_count")

In [25]:
pair_count_matrix_df = read_tcr_epi_pair_counts_csv(pair_count_matrix_csv=Path(project_dir) / "outs" / plates_list_name / f"pair_count_matrix_df_{plates_list_name}.csv")
exp_pair_prob_matrix_df, pair_count_matrix_df = calculate_tcr_epi_pair_exp_prob_matrix(
    pair_count_matrix_df=pair_count_matrix_df,
    normalized_epitope_bulk_counts_df=normalized_epi_bulk_counts_df,
    normalized_tcr_bulk_counts_df=normalized_tcr_bulk_counts_df,
    bulk_epitope_count_col="read_count",
    bulk_tcr_count_col="read_count",
    normalize_using_plate=False,
)

Total called TCR-epitope pair counts: 1839.0
Calculating expected TCR-epitope pair probability matrix using TCR and epitope gDNA bulk sequencing data...
Adding "epi_" prefix to normalized_epitope_bulk_counts_df reference names!
Adding "tcr_" prefix to normalized_epitope_bulk_counts_df reference names!
Epitope reference name(s) in normalized_epitope_bulk_counts_df that are missing in pair_count_matrix_df: {'epi_s_EBV_LMP2_2317_2704', 'epi_s_EBV_BZLF1_2339_2733', 'epi_s_MUC1_1353_1740', 'epi_s_LAGE2_CTAG1B_NY_ESO_1_628_1022', 'epi_s_FLU_NP_2113_2500', 'epi_s_LAGE2_CTAG1B_NY_ESO_1_623_1010', 'epi_s_CMV_IE1_1950_2337', 'epi_s_MAGEB18_400_787', 'epi_s_FLU_NA_2051_2438', 'epi_patient-X_PCNX1_P138L_11'}
Epitope reference name(s) in pair_count_matrix_df that are missing in normalized_epitope_bulk_counts_df: {'epi_patient-X_HSPG2_L276_6', 'epi_patient-X_SLC9A5_E103Q_285', 'epi_patient-X_CCSER2_P329L'}
TCR reference name(s) in normalized_tcr_bulk_counts_df that are missing in pair_count_matrix_d

## 6. Calculate pair fold enrichment, estimate PAIR-Scan confidence in pairs being reactive, and filter potential TCR-TCR and Epi-Epi co-occurrence 

##### Calculate fold enrichment as observed pair frequency / expected pair frequeny.
##### Estimate the probability of observing k sorted pair counts in n sorted wells with an expected pair frequency p.


In [26]:
alpha = 0.05

In [27]:
pair_fold_changes_df = obs_tcr_epi_count_binom_sf_test(
    project_dir=project_dir,
    pair_count_matrix_df=pair_count_matrix_df,
    exp_pair_prob_matrix_df=exp_pair_prob_matrix_df,
    plates_list_name=plates_list_name,
    validating_pair_dict=None,
    fdr_correction=True,
    alpha=alpha, 
    normalize_total_called_pairs=False,
    normalize_total_wells_with_detected_pair=True,
    sort_fold_change=False,
    sort_pvalue=True,
)

1347 TCR-epitope pairs have non-zero well counts of 8736 tested TCR-epitope pairs


In [ ]:
# Typically, only a small number of significant pairs are filtered out by TCR-TCR or Epi-Epi co-occurrence filtering.
# Since transcript_ratio thresholds aren't universal across datasets, we recommend to start at the least stringent threshold
# (1.0) and increase stringency (down to ~0.1) to identify pairs that remain robust vs. those sensitive to co-occurrence.
pair_fold_changes_df = add_tcr_epi_pairs_only_supported_by_co_occurrence_bool_col(
    project_dir=project_dir,
    plates_list_name=plates_list_name,
    pair_fold_changes_df=pair_fold_changes_df,
    epi_transcript_ratio=0.4,  
    tcr_transcript_ratio=0.4,  
)

In [29]:
pair_fold_changes_df = add_pair_entropy_over_plate_col(pair_fold_changes_df = pair_fold_changes_df, project_dir = project_dir)

In [30]:
write_reactive_pair_results(
    pair_fold_changes_df=pair_fold_changes_df,
    project_dir=project_dir,
    plates_list_name=plates_list_name,
)

## 7. Plot screen results

In [ ]:
# backup and re-read so that you can adjust results plots without having to re-run the code above
pair_fold_changes_df.to_excel(Path(project_dir) / "outs" / plates_list_name / f"pair_fold_changes_df_{plates_list_name}.xlsx")
# Run this if needed to re-read without running code above: 
# pair_fold_changes_df = pd.read_excel(Path(project_dir) / "outs" / plates_list_name / f"pair_fold_changes_df_{plates_list_name}.xlsx", index_col = 0)

In [38]:
pair_fold_changes_df["Fold_change"].quantile(0.999)

np.float64(138.95006189658577)

In [43]:
plot_tcr_epi_pair_ma_plot(
    pair_fold_changes_df=pair_fold_changes_df,
    title=plates_list_name,
    xlim_max=0.0015,
    xticks_list=list(np.arange(-0.00001, 0.0015, 0.0005)),
    xlim_min=-0.00001,
    ylim_max=160,
    save_dir=Path(project_dir) / "outs" / plates_list_name,
    custom_pair_color_dict=None,
    plot_filter_only_supported_co_occurrence=True,
    plot_reactive_pair_co_occurrence=False,
    annotate=True,
    annotation_fontsize=2.5,
    w=11,
    h=11,
)

In [44]:
plot_tcr_epi_pair_ma_plot(
    pair_fold_changes_df=pair_fold_changes_df,
    title=plates_list_name,
    xlim_max=0.0015,
    xticks_list=list(np.arange(-0.00001, 0.0015, 0.0005)),
    xlim_min=-0.00001,
    ylim_max=160,
    save_dir=Path(project_dir) / "outs" / plates_list_name,
    custom_pair_color_dict=None,
    plot_filter_only_supported_co_occurrence=True,
    plot_reactive_pair_co_occurrence=False,
    annotate=False,
    # annotation_fontsize = 2.5,
    w=11,
    h=11,
)

In [36]:
plot_pvalue_corrected_rank_vs_pvalue_corrected(
    pair_fold_changes_df=pair_fold_changes_df,
    save_dir=Path(project_dir) / "outs" / plates_list_name,
    title=plates_list_name,
    xlim_max=85,
    xlim_min=-5,
    ylim_max=85,
    ylim_min=-5,
    adj_pval_alpha=None,
    filter_significant_with_more_than_one_epi=True,
    custom_pair_color_dict=None,
    annotate=True,
)

In [37]:
plot_pvalue_corrected_rank_vs_pvalue_corrected(
    pair_fold_changes_df=pair_fold_changes_df,
    save_dir=Path(project_dir) / "outs" / plates_list_name,
    title=plates_list_name,
    xlim_max=85,
    xlim_min=-5,
    ylim_max=85,
    ylim_min=-5,
    adj_pval_alpha=None,
    filter_significant_with_more_than_one_epi=True,
    custom_pair_color_dict=None,
    annotate=False,
)

### Output files in `outs/patient-X/`

**MA plots** — `MA_plot_<filters>_colored[_annotated].pdf` (10 files)
Scatterplot comparing expected pair frequency (in case of random pairing) with pair fold-enrichment (observed/expected pair frequency) (also see Extended Data Fig. 3e). Filter/annotation tags in the filenames mean the following:
- `only_supported_by_co_occurrence` — pair dots are removed if the co-occurrence filter (Extended Data Fig. 3d) found insufficient independent well evidence for that pair (a competing epitope/TCR co-occurred in ≥ threshold fraction of the pair's supporting wells).
- `cross_detected` — pair dots are removed if their TCR is also part of another, more significant pair with a different Ag; only the most significant pair per TCR is kept.
- `significance` — pair dots are colored if `P_value_corrected` (also called "PAIR-Scan confidence in a pair being reactive" in the paper) is below a user-defined FDR significance threshold.
- `validated` — pair dots are colored by individual pair validation co-culture result (annotated in pair_fold_changes_df via `validating_pair_dict`).
- `_annotated` — pair dots are labeled with text describing the pair.

**Rank plots** — `p_value_corrected_vs_p_value_rank_cross_detected_filtered[_co_occurrence_filtered][_only_significance_colored][_annotated].pdf`
Scatterplot comparing -log10(adjusted P values) with their rank (lowest adjusted P value has the highest rank); same filter/annotation tags as above.

**Pair-level tables**
- `pair_counts_sorted_patient-X.{csv,xlsx}` — detected TCR–epitope pairs (long format, across all wells/plates), sorted by number of supporting wells.
  - `well_count`: number of wells the pair was detected in.
- `pair_count_matrix_df_patient-X.{csv,xlsx}` — same pair data as a TCR × epitope well-count matrix.
  - rows: TCR ids, columns: epitope ids, values: well counts.
- `pair_fold_changes_df_patient-X.xlsx` — per-pair statistics.
  - `Fold_change`: observed / expected pair frequency (see Extended Data Fig. 3e).
  - `P_value`, `P_value_corrected`: significance of enrichment (see Extended Data Fig. 3e).
  - `Pearson_residual`: (observed − expected pair count) / √(expected pair count) — a variance-stabilized measure of deviation from the random-pairing null, independent of `Fold_change`.
  - `validating_pair`: True if validating reactivity in individual co-cultures, False otherwise (via `validating_pair_dict`). NaN if no validation co-culture was performed in the wet lab.
  - `BH_Reject`: True if significant, False otherwise.
  - `Exp_pair_prob`: expected pair frequency in case of random pairing.
  - `tcr/epi_only_supported_by_co_occurrence`: True if insufficient independent well evidence, False otherwise.
  - `TCR/epi_significant_with_more_than_one_epi/TCR`: True if significant with more than one Epi or TCR; only the TCR-side flag is used for cross-detection filtering.
  - `pair_plate_entropy`: normalized Shannon entropy (0–1) of the pair's detected-well counts across plates; 0 = all detections concentrated on a single plate, 1 = evenly spread across all plates — flags pairs whose signal is driven by just one plate rather than replicated across the screen, and are thus more likely artefacts.
- `reactive_pair_df_patient-X.xlsx` / `reactive_pair_not_co_occurrence_filtered_df_patient-X.xlsx` — pairs with `P_value_corrected` below the FDR threshold in `pair_fold_changes_df`, with or without the co-occurrence filter applied.
- `cross_detected_df_patient-X.xlsx` — pairs whose TCR is cross-detected with more than one Ag at high confidence; only the most significant pair per TCR is kept, and these are not reported in `reactive_pair_df`.
- `filtered_pair_df_patient-X.xlsx` — pairs removed by the co-occurrence filter and not reported in `reactive_pair_df`.

**Co-occurrence counts** 
- `tcr_co_occurrence_counts_patient-X.{csv,xlsx}` (+ `_total_well_normalized_` variant)
- `epitopes_co_occurrence_counts_patient-X.{csv,xlsx}` (+ `_total_well_normalized_` variant)

**Logs / summaries**
- `summary_statistics_patient-X.txt` — QC summary statistics for the PAIR-Scan screen.
- `reactive_pair_result_patient-X.txt` — `reactive_pair_df` in .txt format; also lists pairs sharing a cross-detected TCR (most significant pair listed first, with its P value).
- `transcript_tuple_well_counter_patient-X.json` — file used for running the pipeline. Do not remove.

## 8. Optional: after co-culture validation of individual TCR-Ag hit pairs, re-run with a validating_pair_dict to color plots by reactive/non-reactive status

In [ ]:
# For example, the validating_pair_dict for the Mel-Pt-B PAIR-Scan screen in the paper
# looks as follows: pairs that validate as 'reactive' are encoded as True, and pairs
# validating as 'non-reactive' are encoded as False in the dictionary.
validating_pair_dict = {
    "epi_mel-pt-b_PGAP6_A52V_182-tcr_1_2_B19_42_mel-pt-b_34": True, 
    "epi_mel-pt-b_BUB1B_L575F_188-tcr_1_2_D14_85_mel-pt-b_77": True,
    "epi_mel-pt-b_BBS12_E612Q_432-tcr_1_2_E4_99_mel-pt-b_91": True, 
    "epi_mel-pt-b_BBS12_E612Q_432-tcr_1_2_A23_22_mel-pt-b_14": True,
    "epi_mel-pt-b_PSMA6_S17L_47-tcr_1_2_C16_63_mel-pt-b_55": True, 
    "epi_mel-pt-b_NBEA_D230N_76-tcr_1_2_D7_78_mel-pt-b_70": False, 

    "epi_mel-pt-b_BICRAL_S805F_339-tcr_1_2_C19_66_mel-pt-b_58": False, 
    "epi_mel-pt-b_OBSCN_E2657K_324-tcr_1_2_C7_54_mel-pt-b_46": False, 
    "epi_mel-pt-b_PKHD1L1_G3255E_508-tcr_1_2_B24_47_mel-pt-b_39": False, 
    "epi_mel-pt-b_PGM2_P507L_252-tcr_1_2_D22_93_mel-pt-b_85": False, 
    "epi_mel-pt-b_EFEMP2_E317DTNRCVE_95-tcr_1_2_D2_73_mel-pt-b_65": False, 
    "epi_mel-pt-b_CDH17_R263W_741-tcr_1_2_D19_90_mel-pt-b_82": False, 
    "epi_mel-pt-b_PIK3CB_D988E_181-tcr_1_2_D8_79_mel-pt-b_71": False, 
    "epi_mel-pt-b_FLT3_296fs_661-tcr_1_2_D20_91_mel-pt-b_83": False, 
    "epi_mel-pt-b_PTPRO_S934F_293-tcr_1_2_D6_77_mel-pt-b_69": False, 
    "epi_mel-pt-b_CSMD2_E1112D_739-tcr_1_2_A12_11_mel-pt-b_3": False, 
    "epi_mel-pt-b_KCNQ5_L302F_627-tcr_1_2_A17_16_mel-pt-b_8": False,

    "epi_mel-pt-b_SCN2A_G266E_489-tcr_1_2_A20_19_mel-pt-b_11": False
    }

### After validation, re-run these with `validating_pair_dict=validating_pair_dict` (instead of `None`) to reflect the validation status in your outputs:
- plot_detected_tcr_epi_plate_map — recolors validated wells in the plate map PDFs.
- write_tcr_epi_pair_well_count_files — re-annotates the pair-calling files with validation status.
- obs_tcr_epi_count_binom_sf_test — re-annotates pair_fold_changes_df/reactive_pair_df with validation status, and colors pair dots by validation status in the downstream MA and P value rank plots.